In [ ]:
# Notebook: Power Spectrum of a Square Pulse Sequence
# Author: Adapted for Springer-style presentation

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, BoundedIntText, FloatSlider

# ==========================
# Parameters
# ==========================

t = np.linspace(-5, 5, 2000)


# ==========================
# Square pulse signal
# ==========================

def x_square(t, A=1, tau=2, T=6):
    t_mod = (t + T/2) % T - T/2
    return np.where(np.abs(t_mod) <= tau/2, A, 0.0)


# ==========================
# Fourier reconstruction (time domain)
# ==========================

def x_fourier_square(t, N, A=1, tau=2, T=6):
    omega = 2 * np.pi / T
    a0 = (A * tau) / T
    xf = np.full_like(t, a0)
    
    for n in range(1, N + 1):
        arg = n * omega * tau / 2
        sinc_val = np.sin(arg) / arg if arg != 0 else 1.0
        an = (A * tau / T) * sinc_val
        xf += 2 * an * np.cos(n * omega * t)
        
    return xf


# ==========================
# Interactive Plot Function
# ==========================

@interact(
    N=BoundedIntText(value=5, min=1, max=50, description="N"),
    A=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description="A"),
    tau=FloatSlider(value=2.0, min=0.5, max=4.0, step=0.1, description="tau"),
    T_slider=FloatSlider(value=6.0, min=2.0, max=10.0, step=0.1, description="T")
)
def plot_square_spectrum(N, A, tau, T_slider):
    if tau >= T_slider:
        print("Σφάλμα: Η περίοδος T πρέπει να είναι μεγαλύτερη από τη διάρκεια tau.")
        return

    omega = 2 * np.pi / T_slider

    # Time domain calculations
    xx = x_square(t, A, tau, T_slider)
    xf = x_fourier_square(t, N, A, tau, T_slider)

    # Frequency domain calculations (Power Spectrum)
    # DC όρος: n = 0, πλάτος a0 = (A*tau)/T, ισχύς = a0^2
    a0 = (A * tau) / T_slider
    dc_power = a0 ** 2

    n_values = np.arange(1, N + 1)
    powers = np.zeros_like(n_values, dtype=float)
    
    for idx, n in enumerate(n_values):
        arg = n * omega * tau / 2
        sinc_val = np.sin(arg) / arg if arg != 0 else 1.0
        an = (A * tau / T_slider) * sinc_val
        # Η κάθε αρμονική στη σειρά έχει πλάτος 2*an, οπότε η ισχύς της είναι (2*an)^2 = 4*an^2
        powers[idx] = (2 * an) ** 2

    # Δημιουργία υπογραφήματος 2 γραμμών
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6.5))

    # --- 1. Time Domain Plot ---
    ax1.plot(t, xx, 'r', linewidth=2.5, label="Original square pulse")
    ax1.plot(t, xf, 'b', linewidth=2, label=f"Fourier expansion (N={N})")
    ax1.set_xlabel(r"$t$")
    ax1.set_ylabel("Amplitude")
    ax1.grid(True)
    ax1.legend(loc='upper right')
    ax1.set_title("Square Pulse Signal Fourier Reconstruction (Time Domain)")

    # --- 2. Power Spectrum Plot (Stem plot με τιμές) ---
    spectrum_n = np.concatenate(([0], n_values))
    spectrum_P = np.concatenate(([dc_power], powers))
    
    markerline, stemlines, baseline = ax2.stem(
        spectrum_n, spectrum_P, 
        basefmt="k-", linefmt='g-', markerfmt='go'
    )
    
    # Εκτύπωση της τιμής ισχύος πάνω από κάθε γραμμή
    for n_val, p_val in zip(spectrum_n, spectrum_P):
        ax2.text(
            n_val, p_val + 0.02 * max(spectrum_P), 
            f"{p_val:.3f}", 
            ha='center', 
            va='bottom', 
            fontsize=9,
            fontweight='bold',
            color='darkgreen'
        )

    ax2.set_xlabel("Harmonic Number ($n$)")
    ax2.set_ylabel("Power ($P_n$)")
    ax2.grid(True)
    ax2.set_title("Discrete Power Spectrum")
    
    # Προσαρμογή ορίων άξονα y
    ax2.set_ylim(0, max(spectrum_P) * 1.15)
    ax2.set_xticks(spectrum_n)

    plt.tight_layout()
    plt.show()
    plt.close(fig)